In [1]:
import colorsys

def hsl_relative_luminance(h, s, l):
    # Convert HSL (with h, s, l in [0,1]) to RGB using colorsys (note: colorsys uses HLS order)
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    
    def srgb_to_linear(c):
        return c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4

    r_lin = srgb_to_linear(r)
    g_lin = srgb_to_linear(g)
    b_lin = srgb_to_linear(b)
    
    # Compute relative luminance using Rec. 709 coefficients
    return 0.2126 * r_lin + 0.7152 * g_lin + 0.0722 * b_lin

# Example usage:
h, s, l = 0.5, 0.5, 0.5
luminance = hsl_relative_luminance(h, s, l)
print(luminance)


0.4222497279399847


In [2]:
# targets = [hsl_relative_luminance(0,0,l/10) for l in range (0,11)]
# targets

In [3]:
import colorsys

def srgb_to_linear(c):
    return c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4

def compute_luminance(l):
    # For h=0, s=0, RGB equals (l, l, l)
    return srgb_to_linear(l)

results = []
current_s = 0.0
targets = [i / 10.0 for i in range(11)]

for target in targets:
    while current_s < 1.0:
        next_s = min(current_s + 0.01, 1.0)
        if compute_luminance(next_s) > target:
            results.append((0, 0, round(current_s, 2)))
            break
        current_s = next_s
    else:
        results.append((0, 0, 1.0))

print(results)


[(0, 0, 0.0), (0, 0, 0.34), (0, 0, 0.48), (0, 0, 0.58), (0, 0, 0.66), (0, 0, 0.73), (0, 0, 0.79), (0, 0, 0.85), (0, 0, 0.9), (0, 0, 0.95), (0, 0, 1.0)]


In [4]:
def draw_swatch(list_of_list, file_name, tile_size):
    rows = len(list_of_list)
    cols = max(len(row) for row in list_of_list)
    width = cols * tile_size
    height = rows * tile_size
    svg = []
    svg.append(f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">')
    
    for i, row in enumerate(list_of_list):
        for j, (h, s, l) in enumerate(row):
            h_deg = h * 360
            s_per = s * 100
            l_per = l * 100
            color = f"hsl({h_deg}, {s_per}%, {l_per}%)"
            x = j * tile_size
            y = i * tile_size
            svg.append(f'<rect x="{x}" y="{y}" width="{tile_size}" height="{tile_size}" fill="{color}" stroke="black" stroke-width="0.25pt" />')
    
    svg.append('</svg>')
    
    with open(file_name, 'w') as f:
        f.write("\n".join(svg))


In [5]:
import colorsys

def srgb_to_linear(c):
    return c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4

def compute_luminance(h, s, l):
    # colorsys uses HLS order: (h, l, s)
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    r_lin = srgb_to_linear(r)
    g_lin = srgb_to_linear(g)
    b_lin = srgb_to_linear(b)
    return 0.2126 * r_lin + 0.7152 * g_lin + 0.0722 * b_lin


def luma_from_hsl(h,s,l):
    r, g, b = colorsys.hls_to_rgb(h, l, s)  # colorsys expects H, L, S.
    return 0.299 * r + 0.587 * g + 0.114 * b

def find_hsl_values_for_target_luminance(h, s, targets, f=compute_luminance):
    results = []
    current_l = 0.5
    for target in targets:
        current_luminance = f(h, s, current_l)
        if current_luminance < target:
            while current_l < 1.0:
                next_l = min(current_l + 0.01, 1.0)
                if f(h, s, next_l) > target:
                    results.append((h, s, round(current_l, 2)))
                    break
                current_l = next_l
            else:
                results.append((h, s, 1.0))
        else:
            while current_l > 0.0:
                next_l = max(current_l - 0.01, 0.0)
                if f(h, s, next_l) < target:
                    results.append((h, s, round(current_l, 2)))
                    break
                current_l = next_l
            else:
                results.append((h, s, 0.0))
    return results


In [6]:
def transpose(matrix):
    return [list(row) for row in zip(*matrix)]

In [7]:

# Example usage:
results = [find_hsl_values_for_target_luminance(0, s/10, [i / 10.0 for i in range(11)], f=luma_from_hsl) for s in range(0,11,2)]

# results = [find_hsl_values_for_target_luminance(0, s, targets, f=luma_from_hsl) for s in [0.0,0.2,0.4,0.6,0.8,1.0]]


In [8]:
draw_swatch(list(reversed(transpose(results))), 'by_lum.svg', 20)

In [9]:
results

[[(0, 0.0, 0.0),
  (0, 0.0, 0.1),
  (0, 0.0, 0.19),
  (0, 0.0, 0.29),
  (0, 0.0, 0.39),
  (0, 0.0, 0.49),
  (0, 0.0, 0.59),
  (0, 0.0, 0.69),
  (0, 0.0, 0.79),
  (0, 0.0, 0.89),
  (0, 0.0, 1.0)],
 [(0, 0.2, 0.0),
  (0, 0.2, 0.1),
  (0, 0.2, 0.21),
  (0, 0.2, 0.32),
  (0, 0.2, 0.43),
  (0, 0.2, 0.53),
  (0, 0.2, 0.62),
  (0, 0.2, 0.72),
  (0, 0.2, 0.81),
  (0, 0.2, 0.9),
  (0, 0.2, 1.0)],
 [(0, 0.4, 0.0),
  (0, 0.4, 0.11),
  (0, 0.4, 0.23),
  (0, 0.4, 0.35),
  (0, 0.4, 0.47),
  (0, 0.4, 0.56),
  (0, 0.4, 0.65),
  (0, 0.4, 0.74),
  (0, 0.4, 0.82),
  (0, 0.4, 0.91),
  (0, 0.4, 1.0)],
 [(0, 0.6, 0.0),
  (0, 0.6, 0.13),
  (0, 0.6, 0.26),
  (0, 0.6, 0.39),
  (0, 0.6, 0.51),
  (0, 0.6, 0.59),
  (0, 0.6, 0.67),
  (0, 0.6, 0.75),
  (0, 0.6, 0.83),
  (0, 0.6, 0.91),
  (0, 0.6, 1.0)],
 [(0, 0.8, 0.0),
  (0, 0.8, 0.14),
  (0, 0.8, 0.29),
  (0, 0.8, 0.44),
  (0, 0.8, 0.54),
  (0, 0.8, 0.62),
  (0, 0.8, 0.69),
  (0, 0.8, 0.77),
  (0, 0.8, 0.84),
  (0, 0.8, 0.92),
  (0, 0.8, 1.0)],
 [(0, 1.0, 0.0),
 

In [10]:
import colorsys
import math

def perceived_saturation(h,s,l):
    # Convert HSL to RGB; colorsys uses HLS order, so pass (h, l, s)
    r, g, b = colorsys.hls_to_rgb(h, l, s)

    r_lin = srgb_to_linear(r)
    g_lin = srgb_to_linear(g)
    b_lin = srgb_to_linear(b)

    # Convert linear RGB to XYZ using the sRGB D65 matrix
    X = 0.4124564 * r_lin + 0.3575761 * g_lin + 0.1804375 * b_lin
    Y = 0.2126729 * r_lin + 0.7151522 * g_lin + 0.0721750 * b_lin
    Z = 0.0193339 * r_lin + 0.1191920 * g_lin + 0.9503041 * b_lin

    # Normalize for the D65 white point
    X /= 0.95047
    Y /= 1.0
    Z /= 1.08883

    # Compute f(t) used in the Lab conversion
    def _f(t):
        return t ** (1/3) if t > 0.008856 else (7.787 * t) + (16/116)

    # Convert to CIELAB
    L_val = 116 * _f(Y) - 16
    a_val = 500 * (_f(X) - _f(Y))
    b_val = 200 * (_f(Y) - _f(Z))

    # Chroma (C*) is used as a measure of perceptual saturation
    chroma = math.sqrt(a_val ** 2 + b_val ** 2)
    return chroma


In [11]:
targets = [perceived_saturation(0,s/10,0.5) for s in range (0,11,2)]
targets

[1.0737672328812771e-05,
 22.20846430445956,
 46.462815230480466,
 69.80665258361117,
 89.93402221805046,
 104.55176567686985]

In [12]:

def find_hsl_values_for_target_chroma(h, l, targets, f=perceived_saturation):
	results = []
	for target in targets:
		current_s = 0.0
		while current_s < 1.0:
			next_s = min(current_s + 0.01, 1.0)
			chr = f(h, next_s, l)
			if chr > target:
				# if (results and results[-1][1] != round(current_s, 2)):
				results.append((h, round(current_s, 2), l))
				break
			current_s = next_s
		else:
			results.append((h, 1.0, l))
			break
		# if results[-1][1] == 1.0:
		# 	break
	return results


In [13]:
find_hsl_values_for_target_chroma(0, 0.5, [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1])

[(0, 0.0, 0.5),
 (0, 0.0, 0.5),
 (0, 0.0, 0.5),
 (0, 0.0, 0.5),
 (0, 0.0, 0.5),
 (0, 0.0, 0.5),
 (0, 0.0, 0.5),
 (0, 0.0, 0.5),
 (0, 0.0, 0.5),
 (0, 0.0, 0.5),
 (0, 0.0, 0.5)]

In [14]:

# Example usage:
results = []
for h in range(0,100):
	results += [find_hsl_values_for_target_chroma(h/100, l/10, range(0,201,15)) for l in range(0,11)]

# results = [find_hsl_values_for_target_luminance(0, s, targets, f=luma_from_hsl) for s in [0.0,0.2,0.4,0.6,0.8,1.0]]


In [15]:
draw_swatch(list(reversed(results)), 'by_lum.svg', 20)

In [16]:
results

[[(0.0, 1.0, 0.0)],
 [(0.0, 0.0, 0.1), (0.0, 0.54, 0.1), (0.0, 1.0, 0.1)],
 [(0.0, 0.0, 0.2),
  (0.0, 0.28, 0.2),
  (0.0, 0.54, 0.2),
  (0.0, 0.86, 0.2),
  (0.0, 1.0, 0.2)],
 [(0.0, 0.0, 0.3),
  (0.0, 0.2, 0.3),
  (0.0, 0.39, 0.3),
  (0.0, 0.58, 0.3),
  (0.0, 0.8, 0.3),
  (0.0, 1.0, 0.3)],
 [(0.0, 0.0, 0.4),
  (0.0, 0.16, 0.4),
  (0.0, 0.31, 0.4),
  (0.0, 0.46, 0.4),
  (0.0, 0.61, 0.4),
  (0.0, 0.79, 0.4),
  (0.0, 1.0, 0.4)],
 [(0.0, 0.0, 0.5),
  (0.0, 0.13, 0.5),
  (0.0, 0.26, 0.5),
  (0.0, 0.38, 0.5),
  (0.0, 0.51, 0.5),
  (0.0, 0.64, 0.5),
  (0.0, 0.8, 0.5),
  (0.0, 1.0, 0.5)],
 [(0.0, 0.0, 0.6),
  (0.0, 0.18, 0.6),
  (0.0, 0.34, 0.6),
  (0.0, 0.5, 0.6),
  (0.0, 0.66, 0.6),
  (0.0, 0.83, 0.6),
  (0.0, 1.0, 0.6)],
 [(0.0, 0.0, 0.7),
  (0.0, 0.24, 0.7),
  (0.0, 0.47, 0.7),
  (0.0, 0.69, 0.7),
  (0.0, 0.91, 0.7),
  (0.0, 1.0, 0.7)],
 [(0.0, 0.0, 0.8), (0.0, 0.38, 0.8), (0.0, 0.73, 0.8), (0.0, 1.0, 0.8)],
 [(0.0, 0.0, 0.9), (0.0, 0.78, 0.9), (0.0, 1.0, 0.9)],
 [(0.0, 0.0, 1.0), (0.0, 1.

In [17]:
reds = [[(0,s/10,l/10) for s in range(0,11,2)] for l in range(0,11)]
reds

[[(0, 0.0, 0.0),
  (0, 0.2, 0.0),
  (0, 0.4, 0.0),
  (0, 0.6, 0.0),
  (0, 0.8, 0.0),
  (0, 1.0, 0.0)],
 [(0, 0.0, 0.1),
  (0, 0.2, 0.1),
  (0, 0.4, 0.1),
  (0, 0.6, 0.1),
  (0, 0.8, 0.1),
  (0, 1.0, 0.1)],
 [(0, 0.0, 0.2),
  (0, 0.2, 0.2),
  (0, 0.4, 0.2),
  (0, 0.6, 0.2),
  (0, 0.8, 0.2),
  (0, 1.0, 0.2)],
 [(0, 0.0, 0.3),
  (0, 0.2, 0.3),
  (0, 0.4, 0.3),
  (0, 0.6, 0.3),
  (0, 0.8, 0.3),
  (0, 1.0, 0.3)],
 [(0, 0.0, 0.4),
  (0, 0.2, 0.4),
  (0, 0.4, 0.4),
  (0, 0.6, 0.4),
  (0, 0.8, 0.4),
  (0, 1.0, 0.4)],
 [(0, 0.0, 0.5),
  (0, 0.2, 0.5),
  (0, 0.4, 0.5),
  (0, 0.6, 0.5),
  (0, 0.8, 0.5),
  (0, 1.0, 0.5)],
 [(0, 0.0, 0.6),
  (0, 0.2, 0.6),
  (0, 0.4, 0.6),
  (0, 0.6, 0.6),
  (0, 0.8, 0.6),
  (0, 1.0, 0.6)],
 [(0, 0.0, 0.7),
  (0, 0.2, 0.7),
  (0, 0.4, 0.7),
  (0, 0.6, 0.7),
  (0, 0.8, 0.7),
  (0, 1.0, 0.7)],
 [(0, 0.0, 0.8),
  (0, 0.2, 0.8),
  (0, 0.4, 0.8),
  (0, 0.6, 0.8),
  (0, 0.8, 0.8),
  (0, 1.0, 0.8)],
 [(0, 0.0, 0.9),
  (0, 0.2, 0.9),
  (0, 0.4, 0.9),
  (0, 0.6, 0.9),
  (0,

In [18]:
reds_chroma = [[(s/10,perceived_saturation(0,s/10,l/10)) for s in range(0,11,2)] for l in range(0,11)]
reds_chroma

[[(0.0, 0.0), (0.2, 0.0), (0.4, 0.0), (0.6, 0.0), (0.8, 0.0), (1.0, 0.0)],
 [(0.0, 3.870268724340149e-06),
  (0.2, 5.540276689989719),
  (0.4, 11.141552084749184),
  (0.6, 16.53946835986247),
  (0.8, 21.804673484501617),
  (1.0, 27.10204585175386)],
 [(0.0, 5.763786785984793e-06),
  (0.2, 10.28096167715096),
  (0.4, 21.560129879596726),
  (0.6, 32.743270458092766),
  (0.8, 42.294131040830756),
  (1.0, 50.49592646798993)],
 [(0.0, 7.510320049762658e-06),
  (0.2, 14.506632518820513),
  (0.4, 30.392530669491297),
  (0.6, 45.91294693736413),
  (0.8, 59.722877022702555),
  (1.0, 70.91752003307538)],
 [(0.0, 9.159776580333757e-06),
  (0.2, 18.45555977458962),
  (0.4, 38.63482447072054),
  (0.6, 58.17321596714356),
  (0.8, 75.24132221930132),
  (1.0, 88.36947923466992)],
 [(0.0, 1.0737672328812771e-05),
  (0.2, 22.20846430445956),
  (0.4, 46.462815230480466),
  (0.6, 69.80665258361117),
  (0.8, 89.93402221805046),
  (1.0, 104.55176567686985)],
 [(0.0, 1.2259388365547943e-05),
  (0.2, 16.76337

In [19]:
def generate_lookup(h):
    return [[{'hsl':(h,s/100,l/100), 'chroma': perceived_saturation(h,s/100,l/100),'luma': luma_from_hsl(h,s/100,l/100)} for l in range(101)] for s in range(101)]

In [20]:
lookup = [[{'hsl':(0,s/100,l/100), 'chroma': perceived_saturation(0,s/100,l/100),'luma': luma_from_hsl(0,s/100,l/100)} for l in range(101)] for s in range(101)]
lookup

[[{'hsl': (0, 0.0, 0.0), 'chroma': 0.0, 'luma': 0.0},
  {'hsl': (0, 0.0, 0.01), 'chroma': 3.2456872624861104e-07, 'luma': 0.01},
  {'hsl': (0, 0.0, 0.02), 'chroma': 6.491374375503872e-07, 'luma': 0.02},
  {'hsl': (0, 0.0, 0.03),
   'chroma': 9.737061637989982e-07,
   'luma': 0.029999999999999995},
  {'hsl': (0, 0.0, 0.04), 'chroma': 1.2982748751007744e-06, 'luma': 0.04},
  {'hsl': (0, 0.0, 0.05), 'chroma': 1.6505078162038403e-06, 'luma': 0.05},
  {'hsl': (0, 0.0, 0.06),
   'chroma': 2.0532323015609385e-06,
   'luma': 0.05999999999999999},
  {'hsl': (0, 0.0, 0.07), 'chroma': 2.508114170868588e-06, 'luma': 0.07},
  {'hsl': (0, 0.0, 0.08), 'chroma': 3.016923622624583e-06, 'luma': 0.08},
  {'hsl': (0, 0.0, 0.09), 'chroma': 3.581347616403157e-06, 'luma': 0.09},
  {'hsl': (0, 0.0, 0.1), 'chroma': 3.870268724340149e-06, 'luma': 0.1},
  {'hsl': (0, 0.0, 0.11),
   'chroma': 4.068767908266886e-06,
   'luma': 0.10999999999999999},
  {'hsl': (0, 0.0, 0.12),
   'chroma': 4.264873967342873e-06,
   '

In [21]:
targets2d = [[{'chroma':chroma,'luma':luma/10} for chroma in range(0,151,15)] for luma in reversed(range(11))]
targets2d

[[{'chroma': 0, 'luma': 1.0},
  {'chroma': 15, 'luma': 1.0},
  {'chroma': 30, 'luma': 1.0},
  {'chroma': 45, 'luma': 1.0},
  {'chroma': 60, 'luma': 1.0},
  {'chroma': 75, 'luma': 1.0},
  {'chroma': 90, 'luma': 1.0},
  {'chroma': 105, 'luma': 1.0},
  {'chroma': 120, 'luma': 1.0},
  {'chroma': 135, 'luma': 1.0},
  {'chroma': 150, 'luma': 1.0}],
 [{'chroma': 0, 'luma': 0.9},
  {'chroma': 15, 'luma': 0.9},
  {'chroma': 30, 'luma': 0.9},
  {'chroma': 45, 'luma': 0.9},
  {'chroma': 60, 'luma': 0.9},
  {'chroma': 75, 'luma': 0.9},
  {'chroma': 90, 'luma': 0.9},
  {'chroma': 105, 'luma': 0.9},
  {'chroma': 120, 'luma': 0.9},
  {'chroma': 135, 'luma': 0.9},
  {'chroma': 150, 'luma': 0.9}],
 [{'chroma': 0, 'luma': 0.8},
  {'chroma': 15, 'luma': 0.8},
  {'chroma': 30, 'luma': 0.8},
  {'chroma': 45, 'luma': 0.8},
  {'chroma': 60, 'luma': 0.8},
  {'chroma': 75, 'luma': 0.8},
  {'chroma': 90, 'luma': 0.8},
  {'chroma': 105, 'luma': 0.8},
  {'chroma': 120, 'luma': 0.8},
  {'chroma': 135, 'luma': 0.8}

In [22]:
def find_closest_matches(lookup, targets2d):
    # Flatten the lookup table.
    flat_lookup = [entry for row in lookup for entry in row]
    result = []
    for target_row in targets2d:
        match_row = []
        for target in target_row:
            best_entry = None
            best_dist = float('inf')
            for entry in flat_lookup:
                dist = (entry['chroma'] - target['chroma'])**2 + (entry['luma'] - target['luma'])**2
                if dist < best_dist:
                    best_dist = dist
                    best_entry = entry
            match_row.append(best_entry)
        result.append(match_row)
    return result


In [23]:
def directed_find_matches(lookup, targets2d):
    result = []
    s_idx = 0
    for target_row in targets2d:
        row_matches = []
        l_idx = 0
        for target in target_row:
            # Increase l until the lookup luma meets or exceeds the target luma.
            while l_idx < 100 and lookup[s_idx][l_idx]['luma'] < target['luma']:
                l_idx += 1
            row_matches.append(lookup[s_idx][l_idx])
            # Increase s until the lookup chroma meets or exceeds the target chroma.
            while s_idx < 100 and lookup[s_idx][l_idx]['chroma'] < target['chroma']:
                s_idx += 1
        result.append(row_matches)
    return result


In [24]:
import math

def binary_search_lookup(lookup, target):
    # Search in the s direction (outer list)
    low_s, high_s = 0, len(lookup) - 1
    best_candidate = None
    best_error = float('inf')
    
    while low_s <= high_s:
        mid_s = (low_s + high_s) // 2
        row = lookup[mid_s]
        # In the current row, do a binary search on l (inner list)
        low_l, high_l = 0, len(row) - 1
        candidate_idx = None
        while low_l <= high_l:
            mid_l = (low_l + high_l) // 2
            current_luma = row[mid_l]['luma']
            if math.isclose(current_luma, target['luma'], rel_tol=1e-3):
                candidate_idx = mid_l
                break
            elif current_luma < target['luma']:
                low_l = mid_l + 1
            else:
                high_l = mid_l - 1
        if candidate_idx is None:
            # No exact match found; choose the index with the smallest luma difference.
            candidates = []
            if low_l < len(row):
                candidates.append(low_l)
            if high_l >= 0:
                candidates.append(high_l)
            candidate_idx = min(candidates, key=lambda idx: abs(row[idx]['luma'] - target['luma']))
        
        candidate = row[candidate_idx]
        error = (candidate['chroma'] - target['chroma'])**2 + (candidate['luma'] - target['luma'])**2
        if error < best_error:
            best_error = error
            best_candidate = candidate
        
        # Use candidate's chroma to direct the s-search.
        if candidate['chroma'] < target['chroma']:
            low_s = mid_s + 1
        else:
            high_s = mid_s - 1

    return best_candidate

def binary_directed_find_matches(lookup, targets2d):
    result = []
    for target_row in targets2d:
        match_row = []
        for target in target_row:
            match = binary_search_lookup(lookup, target)
            match_row.append(match)
        result.append(match_row)
    return result


In [25]:
results = binary_directed_find_matches(lookup, targets2d)
results

[[{'hsl': (0, 0.5, 1.0),
   'chroma': 1.795054880958058e-05,
   'luma': 0.9999999999999999},
  {'hsl': (0, 0.5, 1.0),
   'chroma': 1.795054880958058e-05,
   'luma': 0.9999999999999999},
  {'hsl': (0, 0.5, 1.0),
   'chroma': 1.795054880958058e-05,
   'luma': 0.9999999999999999},
  {'hsl': (0, 0.5, 1.0),
   'chroma': 1.795054880958058e-05,
   'luma': 0.9999999999999999},
  {'hsl': (0, 0.5, 1.0),
   'chroma': 1.795054880958058e-05,
   'luma': 0.9999999999999999},
  {'hsl': (0, 0.5, 1.0),
   'chroma': 1.795054880958058e-05,
   'luma': 0.9999999999999999},
  {'hsl': (0, 0.5, 1.0),
   'chroma': 1.795054880958058e-05,
   'luma': 0.9999999999999999},
  {'hsl': (0, 0.5, 1.0),
   'chroma': 1.795054880958058e-05,
   'luma': 0.9999999999999999},
  {'hsl': (0, 0.5, 1.0),
   'chroma': 1.795054880958058e-05,
   'luma': 0.9999999999999999},
  {'hsl': (0, 0.5, 1.0),
   'chroma': 1.795054880958058e-05,
   'luma': 0.9999999999999999},
  {'hsl': (0, 0.5, 1.0),
   'chroma': 1.795054880958058e-05,
   'luma'

In [26]:
results = [[item['hsl'] for item in row] for row in results]
results

[[(0, 0.5, 1.0),
  (0, 0.5, 1.0),
  (0, 0.5, 1.0),
  (0, 0.5, 1.0),
  (0, 0.5, 1.0),
  (0, 0.5, 1.0),
  (0, 0.5, 1.0),
  (0, 0.5, 1.0),
  (0, 0.5, 1.0),
  (0, 0.5, 1.0),
  (0, 0.5, 1.0)],
 [(0, 0.0, 0.9),
  (0, 1.0, 0.93),
  (0, 1.0, 0.93),
  (0, 1.0, 0.93),
  (0, 1.0, 0.93),
  (0, 1.0, 0.93),
  (0, 1.0, 0.93),
  (0, 1.0, 0.93),
  (0, 1.0, 0.93),
  (0, 1.0, 0.93),
  (0, 1.0, 0.93)],
 [(0, 0.0, 0.8),
  (0, 0.46, 0.83),
  (0, 0.94, 0.85),
  (0, 0.94, 0.85),
  (0, 0.94, 0.85),
  (0, 0.94, 0.85),
  (0, 0.94, 0.85),
  (0, 0.94, 0.85),
  (0, 0.94, 0.85),
  (0, 0.94, 0.85),
  (0, 0.94, 0.85)],
 [(0, 0.0, 0.7),
  (0, 0.28, 0.73),
  (0, 0.61, 0.76),
  (0, 0.97, 0.78),
  (0, 0.97, 0.78),
  (0, 0.97, 0.78),
  (0, 0.97, 0.78),
  (0, 0.97, 0.78),
  (0, 0.97, 0.78),
  (0, 0.97, 0.78),
  (0, 0.97, 0.78)],
 [(0, 0.0, 0.6),
  (0, 0.2, 0.63),
  (0, 0.42, 0.66),
  (0, 0.65, 0.68),
  (0, 0.95, 0.71),
  (0, 1.0, 0.71),
  (0, 1.0, 0.71),
  (0, 1.0, 0.71),
  (0, 1.0, 0.71),
  (0, 1.0, 0.71),
  (0, 1.0, 0.71)

In [27]:
def remove_repeating_final_entries(result):
    new_result = []
    for row in result:
        new_row = row[:]
        while len(new_row) > 1 and new_row[-1] == new_row[-2]:
            new_row.pop()
        new_result.append(new_row)
    return new_result

# Example usage:
example = [['a','b','c','d','e','e','e','e','e']]
print(remove_repeating_final_entries(example))


[['a', 'b', 'c', 'd', 'e']]


In [28]:
results = remove_repeating_final_entries(results)
results

[[(0, 0.5, 1.0)],
 [(0, 0.0, 0.9), (0, 1.0, 0.93)],
 [(0, 0.0, 0.8), (0, 0.46, 0.83), (0, 0.94, 0.85)],
 [(0, 0.0, 0.7), (0, 0.28, 0.73), (0, 0.61, 0.76), (0, 0.97, 0.78)],
 [(0, 0.0, 0.6),
  (0, 0.2, 0.63),
  (0, 0.42, 0.66),
  (0, 0.65, 0.68),
  (0, 0.95, 0.71),
  (0, 1.0, 0.71)],
 [(0, 0.0, 0.5),
  (0, 0.15, 0.53),
  (0, 0.3, 0.55),
  (0, 0.48, 0.58),
  (0, 0.68, 0.61),
  (0, 0.91, 0.63),
  (0, 1.0, 0.64)],
 [(0, 0.0, 0.4),
  (0, 0.16, 0.43),
  (0, 0.29, 0.45),
  (0, 0.4, 0.48),
  (0, 0.51, 0.5),
  (0, 0.69, 0.53),
  (0, 0.91, 0.56),
  (0, 1.0, 0.57)],
 [(0, 0.0, 0.3),
  (0, 0.19, 0.32),
  (0, 0.35, 0.35),
  (0, 0.49, 0.37),
  (0, 0.62, 0.4),
  (0, 0.74, 0.43),
  (0, 0.88, 0.46),
  (0, 1.0, 0.5)],
 [(0, 0.0, 0.2),
  (0, 0.26, 0.22),
  (0, 0.46, 0.25),
  (0, 0.64, 0.27),
  (0, 0.81, 0.3),
  (0, 0.98, 0.33),
  (0, 1.0, 0.33)],
 [(0, 0.0, 0.1), (0, 0.44, 0.12), (0, 0.77, 0.14), (0, 1.0, 0.17)],
 [(0, 0.5, 0.0)]]

In [29]:
hues = [
	357/360,
	 10/360,
	 22/360,
	 32/360,
	 43/360,
	 51/360,
	 60/360,
	 77/360,
	102/360,
	139/360,
	161/360,
	172/360,
	183/360,
	188/360,
	194/360,
	197/360,
	204/360,
	214/360,
	225/360,
	248/360,
	274/360,
	299/360,
	322/360,
	337/360,
]

In [30]:
hues = [
	357/360,
	337/360,
	312/360,
	291/360,
	268/360,
	239/360,
	212/360,
	195/360,
	178/360,
	144/360,
	121/360,
	 87/360,
	 49/360,
	 39/360,
	 30/360,
	 11/360,
]

In [34]:
hues = [
	357/360,
	337/360,
	322/360,
	299/360,
	274/360,
	248/360,
	225/360,
	214/360,
	204/360,
	197/360,
	194/360,
	188/360,
	183/360,
	172/360,
	161/360,
	139/360,
	102/360,
	77/360,
	60/360,
	51/360,
	43/360,
	32/360,
	22/360,
	10/360,
]

In [35]:
# hues = [h/40 for h in range(40)]
results = []
for h in hues:
	lookup = generate_lookup(h)
	result = binary_directed_find_matches(lookup, targets2d)
	result = [[item['hsl'] for item in row] for row in result]
	result = remove_repeating_final_entries(result)
	results += result
 


In [36]:
results

[[(0.9916666666666667, 0.5, 1.0)],
 [(0.9916666666666667, 0.0, 0.9), (0.9916666666666667, 1.0, 0.93)],
 [(0.9916666666666667, 0.0, 0.8),
  (0.9916666666666667, 0.46, 0.83),
  (0.9916666666666667, 0.97, 0.85)],
 [(0.9916666666666667, 0.0, 0.7),
  (0.9916666666666667, 0.28, 0.73),
  (0.9916666666666667, 0.61, 0.76),
  (0.9916666666666667, 0.99, 0.78),
  (0.9916666666666667, 1.0, 0.78)],
 [(0.9916666666666667, 0.0, 0.6),
  (0.9916666666666667, 0.2, 0.63),
  (0.9916666666666667, 0.42, 0.66),
  (0.9916666666666667, 0.66, 0.68),
  (0.9916666666666667, 0.97, 0.71),
  (0.9916666666666667, 1.0, 0.71)],
 [(0.9916666666666667, 0.0, 0.5),
  (0.9916666666666667, 0.15, 0.53),
  (0.9916666666666667, 0.3, 0.55),
  (0.9916666666666667, 0.48, 0.58),
  (0.9916666666666667, 0.68, 0.6),
  (0.9916666666666667, 0.93, 0.63),
  (0.9916666666666667, 1.0, 0.64)],
 [(0.9916666666666667, 0.0, 0.4),
  (0.9916666666666667, 0.16, 0.43),
  (0.9916666666666667, 0.29, 0.45),
  (0.9916666666666667, 0.41, 0.48),
  (0.9916

In [37]:
draw_swatch(results, "lch_munsell.svg", 20)